# core

> Using pyinstrument to profile FastHTML apps.

Sometimes when building FastHTML apps we run into performance bottlenecks. Figuring out what is slow can be challenging, especially when building apps with async components. That's where profiling tools like pyinstrument can help. Profilers are tools that show exactly how long each component of a project takes to run. Identifying slow parts of an app is the first step in figuring out how to make things run faster.

In [ ]:
#| default_exp core

In [ ]:
#| export
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware import Middleware
from functools import wraps
try: from rich import print
except: pass
try:
    from pyinstrument import Profiler
except ImportError:
    raise ImportError('Please install pyinstrument')

In [ ]:
#| export
class ProfileMiddleware(BaseHTTPMiddleware):    
    async def dispatch(self, request, call_next):
        profiling = request.query_params.get("profile", False)
        term = request.query_params.get("term", False)  
        if profiling:
            profiler = Profiler()
            profiler.start()
            response = await call_next(request)            
            profiler.stop()
            if term: 
                print(profiler.output_text())
                return response
            return HTMLResponse(profiler.output_html())
        return await call_next(request)

In [ ]:
#| export
def instrument(route_handler):
    @functools.wraps(route_handler)
    def _wrapper(*args, **kwargs):
        # Profile the action
        profiler = Profile()
        profiler.start()
        route_handler(*args, **kwargs)
        profiler.stop()
        # Return the profile output
        return profiler.output_html()
    return _wrapper

## Tests

In [ ]:
from starlette.testclient import TestClient
from fastcore.all import *
from httpx import ASGITransport, AsyncClient
from functools import partialmethod
from anyio import from_thread

In [ ]:
class Client:
    "A simple httpx ASGI client that doesn't require `async`"
    def __init__(self, app, url="http://testserver"):
        self.cli = AsyncClient(transport=ASGITransport(app), base_url=url)

    def _sync(self, method, url, **kwargs):
        async def _request(): return await self.cli.request(method, url, **kwargs)
        with from_thread.start_blocking_portal() as portal: return portal.call(_request)

for o in ('get', 'post', 'delete', 'put', 'patch', 'options'): setattr(Client, o, partialmethod(Client._sync, o))

In [ ]:
app, rt = fast_app()
app.add_middleware(ProfileMiddleware)
client = Client(app)

First, confirm that the view works normally

In [ ]:
@rt
def index(): return Titled('Hello, profiler')
assert 'Hello, profiler' in client.get('/').text

Now lets profile it! Or rather, check that it works.

In [ ]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

Let's print to the terminal

In [ ]:
client.get('/?profile=1&term=1')

_     ._   __/__   _ _  _  _ _/_   Recorded: 10:22:10  Samples:  3
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.004     CPU time: 0.004
/   _/                      v5.0.1

Profile at /var/folders/3k/p8fttyg52s30bj9vdr1v7c180000gn/T/ipykernel_48818/207799178.py:8

0.003 Handle._run  asyncio/events.py:86
`- 0.003 coro  starlette/middleware/base.py:135
      [17 frames hidden]  starlette, fasthtml, fastcore
         0.003 Router.__call__  starlette/routing.py:710
         |  0.002 app  starlette/routing.py:69
         |  `- 0.001   starlette/routing.py
         |     0.001 isinstance  <built-in>
         `- 0.001   starlette/routing.py

<Response [200 OK]>

In [ ]:
@instrument
@rt
def saxaphone(): return Titled('Play that sweet horn')
client.get('/saxaphone').text

' <!doctype html>\n <html>\n   <head>\n     <title>Play that sweet horn</title>\n     <meta charset="utf-8">\n     <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n<script src="https://unpkg.com/htmx.org@2.0.4/dist/htmx.min.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js"></script><script src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js"></script>     <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css">\n     <style>:root { --pico-font-size: 100%; }</style>\n<script>\n    function sendmsg() {\n        window.parent.postMessage({height: document.documentElement.offsetHeight}, \'*\');\n    }\n    window.onload = function() {\n        sendmsg();\n        document.body.addEventListener(\'htmx:afterSettle\',    sendmsg);\n        document.body.addEvent

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()